# Run_BrickScanner

Purpose: Ad hoc BrickScanner execution with configurable parameters.

**Widgets:**
- `include_paths`: Comma-separated workspace paths to scan
- `dry_run`: Whether to run in dry-run mode (returns results without persisting)
- `run_setup`: Whether to run DB_Setup before scanning

## Define Widgets

In [0]:
%sh 
pip install -q langchain-text-splitters

In [0]:
%sh
pip --disable-pip-version-check install -c /databricks/.core_packages/immutable-package-constraints.txt -r requirements.txt
dbutils.library.restartPython()

  Using cached databricks_sdk-0.12.0-py3-none-any.whl.metadata (35 kB)
  Using cached langchain-0.2.1-py3-none-any.whl.metadata (13 kB)
  Using cached pydantic-2.5.3-py3-none-any.whl.metadata (65 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (62 kB)
  Using cached tenacity-8.5.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached pydantic_core-2.14.6-cp312-cp312-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (6.5 kB)
INFO: pip is looking at multiple versions of mlflow-skinny to determine which version is compatible with other requirements. This could take a while.
  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-2.22.4-py3-none-any.whl.metadata (31 kB)
  Using cac

In [0]:
# Define widgets
dbutils.widgets.text("include_paths", "/Workspace", "Include Paths (comma-separated)")
dbutils.widgets.dropdown("dry_run", "False", ["True", "False"], "Dry Run (returns results, no persist)")
dbutils.widgets.dropdown("run_setup", "False", ["False", "True"], "Run DB_Setup first")

include_paths = dbutils.widgets.get("include_paths")
dry_run = dbutils.widgets.get("dry_run").lower() == "true"
run_setup = dbutils.widgets.get("run_setup").lower() == "true"

## Install Dependencies

## Setup Imports and Configuration

In [0]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("[Run_BrickScanner]")

## Validate Inputs

In [0]:
# Validate inputs
if not include_paths or not include_paths.strip():
    raise ValueError("include_paths cannot be empty")

print(f"[Run_BrickScanner] Configuration:")
print(f"  Include Paths: {include_paths}")
print(f"  Dry Run: {dry_run}")
print(f"  Run Setup: {run_setup}")

[Run_BrickScanner] Configuration:
  Include Paths: /Workspace/Users/saikiranreddykalluri@gmail.com/Ai Explore/Quick Demo
  Dry Run: False
  Run Setup: False


## Run DB_Setup if Requested

In [0]:
# Run setup if requested
if run_setup:
    print("[Run_BrickScanner] Running DB_Setup...")
    dbutils.notebook.run("./DB_Setup",6000)
    print("[Run_BrickScanner] DB_Setup completed")

## Execute Scanner

In [0]:
# Import and execute scanner
import logging
logger = logging.getLogger("[Run_BrickScanner]")

print("[Run_BrickScanner] Importing BrickScanner package...")
import sys
import importlib

# Remove brickscanner from cache if already loaded
if 'brickscanner' in sys.modules:
    print("[Run_BrickScanner] Reloading brickscanner module...")
    import brickscanner
    from brickscanner import scanner
    importlib.reload(scanner)
    importlib.reload(brickscanner)
else:
    import brickscanner

print("[Run_BrickScanner] Starting scan...")
try:
    result_df = brickscanner.run(
        spark,
        extractor_type="genai",
        include_paths=include_paths,
        dry_run=dry_run
    )
    
    if dry_run:
        print("[Run_BrickScanner] ✓ Dry run completed")
        display(result_df)
    else:
        print("[Run_BrickScanner] ✓ Scan completed and results persisted")
        
        # Query and display summary
        from brickscanner.config import DB_TABLES
        output_table = DB_TABLES["genai_output"]
        
        summary_df = spark.sql(f"""
            SELECT 
                severity,
                COUNT(*) as count
            FROM {output_table}
            GROUP BY severity
            ORDER BY 
                CASE 
                    WHEN severity = '1-High' THEN 1
                    WHEN severity = '2-Medium' THEN 2
                    WHEN severity = '3-Low' THEN 3
                    ELSE 4
                END
        """)
        
        print("[Run_BrickScanner] Summary:")
        display(summary_df)
        
        total = spark.sql(f"SELECT COUNT(*) as total FROM {output_table}").collect()[0].total
        print(f"[Run_BrickScanner] Total findings: {total}")

except Exception as e:
    logger.error(f"Scan failed: {e}")
    raise

INFO:[scanner]:[scanner] Starting BrickScanner run (dry_run=False)
INFO:[scanner]:[scanner] Scanning paths: ['/Workspace/Users/saikiranreddykalluri@gmail.com/Ai Explore/Quick Demo']


[Run_BrickScanner] Importing BrickScanner package...
[Run_BrickScanner] Reloading brickscanner module...
[Run_BrickScanner] Starting scan...


In [0]:
!pip install langchain_databricks
!pip install langchain

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.3.14 requires langchain-core<2.0.0,>=1.4.9, but you have langchain-core 0.3.86 which is incompatible.
langchain-text-splitters 1.1.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langchain-openai 1.1.6 requires langchain-core<2.0.0,>=1.2.2,